# Module 12 — Design Principles in Python

## Exercise 12.2 — Four descriptors

Run:  python ex02_descriptors.py

---

**How to work through this.** Each task below is its own cell. Run them one at a
time and read the output before moving on; that is the whole advantage of a
notebook over a script. Where a cell asks for a prediction, write it before you
run anything. Being wrong on purpose in a place where it costs nothing is how
the correct model gets built.

---

# The concepts behind this exercise

Read this before the tasks. Every idea the tasks below use is explained here, so
you should not need to leave this notebook.

The code cells in this part are demonstrations rather than exercises. Run them,
change a value, run them again. That is the whole point of having them here
instead of in a document.

## Concept 1. SOLID, translated

### S — Single responsibility

*A module should have one reason to change.* Survives intact, and applies to
functions and modules at least as much as to classes.

The practical test is not "does this class do one thing" (unanswerably vague) but
**"who asks for changes to this?"** If the finance team and the marketing team
both file tickets against one class, it has two responsibilities.

In [ ]:
class Report:                     # three reasons to change
    def fetch(self): ...          # the database team
    def calculate(self): ...      # the finance team
    def render_pdf(self): ...     # the design team

### O — Open/closed

*Open for extension, closed for modification.* In Java this means inheritance and
interfaces. **In Python it usually means a function argument.**

In [ ]:
# closed for modification: you never edit this
def process(items, transform=lambda x: x, key=None, on_error=None): ...

# extension is a registry, not a subclass
HANDLERS: dict[str, Callable[[Event], None]] = {}

def handles(event_type: str):
    def register(fn):
        HANDLERS[event_type] = fn
        return fn
    return register

@handles("click")
def on_click(event): ...

That registry is the Strategy pattern, the Command pattern, and half of the
Visitor pattern, in eight lines and with no classes.

### L — Liskov substitution

Covered in Module 10, and it applies unchanged. Python's dynamism makes it easier
to violate and no less costly when you do — the failure just moves from compile
time to runtime.

### I — Interface segregation

*No client should depend on methods it does not use.* This is where `Protocol`
shines (Module 10): define the **narrowest** interface your function actually
needs.

In [ ]:
class Readable(Protocol):
    def read(self, n: int) -> bytes: ...

def parse(source: Readable) -> Document: ...   # not "a File", just "readable"

Now a file, a socket, a `BytesIO`, and a test fake all qualify. Typing the
parameter as a concrete class instead would have excluded three of them for no
reason.

### D — Dependency inversion

*Depend on abstractions, not concretions.* In Python, the abstraction is usually
a **function parameter**, not an interface hierarchy:

In [ ]:
# concrete dependency: untestable without a database and a clock
class OrderService:
    def __init__(self):
        self.db = PostgresConnection("prod")
        self.clock = datetime.now

# inverted: the caller supplies both
class OrderService:
    def __init__(self, db: SupportsQuery, now: Callable[[], datetime] = datetime.now):
        self.db = db
        self.now = now

That second version is testable with a dict and a lambda. No framework, no
container, no annotations — this **is** dependency injection, and in Python it
needs no library.

---

## Concept 2. Patterns that Python dissolves

| Pattern | In Java | In Python |
|---|---|---|
| **Strategy** | An interface + N classes | Pass a function |
| **Command** | An interface + N classes | Pass a function, or `partial` |
| **Factory** | A factory class | A function, or a `@classmethod` |
| **Abstract Factory** | Two class hierarchies | A dict of constructors |
| **Singleton** | Private ctor + static instance | A module. Modules are singletons. |
| **Decorator** | Wrapper class hierarchy | `@decorator` (Module 15) |
| **Observer** | Listener interfaces | A list of callables |
| **Iterator** | An interface | `__iter__` / `yield` (Module 14) |
| **Template Method** | Abstract base + hooks | A function taking hook functions |
| **Adapter** | A wrapper class | Often just duck typing |
| **Builder** | A builder class | Keyword arguments, or a frozen dataclass + `replace` |
| **Visitor** | Double dispatch | `match`, or `functools.singledispatch` |

The patterns that remain useful in Python — because they solve a real structural
problem rather than a missing language feature — are Adapter (when the interfaces
genuinely differ), Facade, Proxy, Repository, and Unit of Work.

**Singleton deserves a note**, because it is the one people reach for most and
need least:

In [ ]:
# config.py
_settings = load_settings()

def get_settings() -> Settings:
    return _settings

A module is imported once per process and cached in `sys.modules` (Module 06).
That is a singleton, with none of the thread-safety problems of the
double-checked-locking version and none of the testability problems of a class
that hides its own construction.

---

## Concept 4. Descriptors

The mechanism under `@property`, `@classmethod`, `@staticmethod`,
`cached_property`, and every ORM field you have ever used.

In [ ]:
class Positive:
    """A reusable validated attribute."""

    def __set_name__(self, owner: type, name: str) -> None:
        self._name = f"_{name}"          # called at CLASS creation time

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self                   # accessed on the CLASS, not an instance
        return getattr(obj, self._name)

    def __set__(self, obj, value) -> None:
        if value <= 0:
            raise ValueError(f"{self._name[1:]} must be positive, got {value}")
        setattr(obj, self._name, value)


class Product:
    price = Positive()          # written ONCE
    weight = Positive()
    quantity = Positive()

Three properties would have been thirty lines of near-identical code. The
descriptor is written once and reused.

**Data versus non-data descriptors** (Module 08, exercise 1): defining both
`__get__` and `__set__` makes it a *data* descriptor, which takes priority over
the instance `__dict__`. Defining only `__get__` makes it *non-data*, which the
instance dict beats — and that asymmetry is precisely how `cached_property`
works.

**When to use a descriptor:** the same attribute logic repeated across three or
more attributes, or across several classes. Below that, `@property` is clearer.

---

## Concept 6. Metaclasses

> "Metaclasses are deeper magic than 99% of users should ever worry about. If you
> wonder whether you need them, you don't." — Tim Peters

A metaclass is the class of a class. `type` is the default.

In [ ]:
class Meta(type):
    def __new__(mcls, name, bases, namespace, **kwargs):
        namespace["created_by"] = "Meta"
        return super().__new__(mcls, name, bases, namespace)

class Thing(metaclass=Meta): ...
Thing.created_by            # 'Meta'

**Use one only when you must change how the class object itself is created** —
before it exists. In practice that means: ABCs (`ABCMeta`), enums (`EnumMeta`),
and ORM/serialization frameworks that rewrite the class namespace. Django models,
SQLAlchemy declarative, and Pydantic v1 all use them.

Everything else is better served by:

| Want | Use |
|---|---|
| React to subclass creation | `__init_subclass__` |
| Modify a class after creation | A class decorator |
| Reusable attribute behaviour | A descriptor |
| Prevent instantiation | ABC with `@abstractmethod` |
| Enforce an interface | `Protocol` + a type checker |
| Register implementations | `__init_subclass__` or a decorator |

Two costs worth knowing: metaclass conflicts (a class cannot inherit from two
classes with unrelated metaclasses) and the fact that they defeat most readers'
ability to follow the code.

---

---

# Now the exercise

You have everything you need. Work top to bottom, and where a cell asks for a
prediction, write it before you run anything.

## The concepts this exercise uses

These are the numbered sections of [the module README](../README.md). If a task below stops making sense, the section named next to it is the one to re-read.

- Section 1: SOLID, translated
- Section 2: Patterns that Python dissolves
- Section 3: Composition over inheritance, concretely
- Section 4: Descriptors
- Section 5: `__init_subclass__` and class decorators
- Section 6: Metaclasses
- Section 7: When not to use a class at all

> The teaching for this module currently lives in the README rather than in this notebook. Read it alongside these cells.

## Setup

Run this first. It is the imports and any shared values the tasks below need.

In [ ]:
from __future__ import annotations

from typing import Any


# TODO 1 -----------------------------------------------------------------------

---

## `Positive`

A validated numeric attribute that must be > 0.

In [ ]:
class Positive:
    """A validated numeric attribute that must be > 0.

    Implement __set_name__, __get__, __set__.
    __get__ must return the descriptor itself when accessed on the CLASS
    (obj is None) -- otherwise Product.price would raise, and help(), Sphinx,
    and every introspection tool would break.
    """

---

## `Typed`

Runtime type enforcement: Typed(str), Typed(int), Typed(list, str).

In [ ]:
class Typed:
    """Runtime type enforcement: Typed(str), Typed(int), Typed(list, str).

    Then answer: this is what type hints DO NOT do (Module 04). When is runtime
    enforcement worth it, and when is a static checker enough?
    """

---

## `Lazy`

Compute once, on first access, then cache -- functools.cached_property,

In [ ]:
class Lazy:
    """Compute once, on first access, then cache -- functools.cached_property,
    written by hand.

    The whole trick is ONE line: define __get__ but NOT __set__, so this is a
    NON-DATA descriptor and the instance __dict__ beats it. On first access,
    write the computed value into obj.__dict__[name]. Every later access finds
    it at rung 2 of the lookup ladder and never reaches this class again.

    Prove it: put a print in the computation and access the attribute three
    times. Then check that the value really is in obj.__dict__.

    Then answer: what breaks if you add a __set__ method that just raises?
    (Try it. The result is instructive and is exactly why cached_property has
    the shape it does.)
    """

---

## `Unit`

A quantity with a unit, converting on assignment.

In [ ]:
class Unit:
    """A quantity with a unit, converting on assignment.

        class Recipe:
            flour = Unit("g", {"kg": 1000, "g": 1, "oz": 28.35})

        r.flour = "2 kg"     ->  stored as 2000 (grams)
        r.flour              ->  2000
        r.flour_display      ->  "2000 g"

    Accept a number (assumed to be in the canonical unit) or a string with a
    unit suffix. Reject unknown units with a message listing the valid ones.
    """

---

## `ProductBefore`

Six properties, thirty lines, four of them identical apart from a name.

In [ ]:
class ProductBefore:
    """Six properties, thirty lines, four of them identical apart from a name.
    This is the duplication descriptors remove."""

    def __init__(self, name: str, price: float, weight: float) -> None:
        self.name = name
        self.price = price
        self.weight = weight

    @property
    def price(self) -> float:
        return self._price

    @price.setter
    def price(self, value: float) -> None:
        if value <= 0:
            raise ValueError("price must be positive")
        self._price = value

    @property
    def weight(self) -> float:
        return self._weight

    @weight.setter
    def weight(self, value: float) -> None:
        if value <= 0:
            raise ValueError("weight must be positive")
        self._weight = value

---

## `verify`

_verify_

In [ ]:
def verify() -> None:
    class Product:
        name = Typed(str)                        # type: ignore[name-defined]
        price = Positive()                       # type: ignore[name-defined]
        weight = Positive()                      # type: ignore[name-defined]
        tags = Typed(list)                       # type: ignore[name-defined]

        def __init__(self, name: str, price: float, weight: float) -> None:
            self.name, self.price, self.weight = name, price, weight
            self.tags = []

        @Lazy                                     # type: ignore[name-defined]
        def expensive_score(self) -> float:
            print("      (computing score)")
            return self.price * self.weight

    p = Product("widget", 10.0, 2.0)
    assert p.price == 10.0

    for attr, bad in [("price", 0), ("weight", -1)]:
        try:
            setattr(p, attr, bad)
        except ValueError:
            pass
        else:
            raise AssertionError(f"{attr} accepted {bad}")

    try:
        p.name = 123          # type: ignore[assignment]
    except TypeError:
        pass
    else:
        raise AssertionError("Typed must reject the wrong type")

    assert isinstance(Product.price, Positive), (   # type: ignore[name-defined]
        "__get__ must return self when accessed on the class"
    )

    print("    accessing expensive_score three times:")
    assert p.expensive_score == 20.0
    assert p.expensive_score == 20.0
    assert p.expensive_score == 20.0
    assert "expensive_score" in p.__dict__, "Lazy must cache into the instance dict"

    class Recipe:
        flour = Unit("g", {"kg": 1000, "g": 1, "oz": 28.35})   # type: ignore[name-defined]

    r = Recipe()
    r.flour = "2 kg"          # type: ignore[assignment]
    assert r.flour == 2000
    r.flour = 500             # type: ignore[assignment]
    assert r.flour == 500
    try:
        r.flour = "2 furlongs"   # type: ignore[assignment]
    except ValueError as exc:
        assert "kg" in str(exc), "the error must list the valid units"
    else:
        raise AssertionError("unknown unit must be rejected")

    print("all descriptor checks passed")

---

## Run it

This is what running the original file did. Everything above must have been run first.

In [ ]:
if __name__ == "__main__":
    verify()

---

## Before you move on

- [ ] Every cell above ran, in order, on a fresh kernel.
- [ ] You wrote a prediction before running, wherever one was asked for.
- [ ] You can say in one sentence what each task was actually testing.
- [ ] Anything that surprised you is written down in `PROGRESS.md`.

Compare against the worked answers in `../solutions/` only after your own
attempt runs.